# Deliverable 4 – Temperatures and Fractions

**Group:** Kristian Sheath and Philip Schou Whitley<br>
**Repository:** <https://github.com/SheathAndWhitley/inf201_deliverable_4><br>

This notebook presents our two classes, shows them in use and includes the `pytest` output for each.

| Module | Class | Tests |
|---|---|---|
| `temperature.py` | `Temperature` – a temperature in °C, °F or K | `test_temperature.py` (30 tests) |
| `fraction.py` | `Fraction` – an exact fraction of two integers | `test_fraction.py` (7 tests) |

Our use of generative AI is described in `GenAI_Use.md`, and the commit log is in `git_log.txt`.

In [1]:
import sys

from temperature import Temperature
from fraction import Fraction

## Task 1: `Temperature`

**Read-only attributes:** `value`, `unit`<br>
**Conversion methods:** `to_celsius()`, `to_fahrenheit()`, `to_kelvin()`, `convert_to(target_unit)`<br>
**Special methods:** `__repr__`, `__str__`, `__eq__`, `__ne__`, `__lt__`, `__le__`, `__gt__`, `__ge__`<br>

Design choices:
* The unit is upper-cased on creation, so `"c"` and `"C"` are the same. The default unit is `"C"`.
* `value` and `unit` are stored as `_value` and `_unit` and exposed through `@property` with no setter, so they can't be changed after creation.
* All comparisons convert both temperatures to kelvin first, so different units can be compared. Equality uses a tolerance of `1e-9` because float conversions are not exact (e.g. 98.6 °F → °C).
* Comparing with something that isn't a `Temperature` returns `NotImplemented`, which lets Python raise a `TypeError` for ordering (and return `False`/`True` for `==`/`!=`).

### Creating temperatures and converting between units

In [2]:
t = Temperature(25)
print(repr(t), "->", t)

body = Temperature(98.6, "f")
print(repr(body), "->", body)
print(f"  to_celsius()    = {body.to_celsius():.2f}")
print(f"  to_kelvin()     = {body.to_kelvin():.2f}")
print(f"  convert_to('K') = {body.convert_to('k')}")

Temperature(25.0, 'C') -> 25.00 °C
Temperature(98.6, 'F') -> 98.60 °F
  to_celsius()    = 37.00
  to_kelvin()     = 310.15
  convert_to('K') = 310.15 K


### Invalid input

In [3]:
for args in [(0, "X"), (-300, "C"), (-500, "F"), (-1, "K")]:
    try:
        Temperature(*args)
    except ValueError as error:
        print(f"Temperature{args}: ValueError: {error}")

try:
    t.value = 30
except AttributeError as error:
    print("t.value = 30: AttributeError:", error)

Temperature(0, 'X'): ValueError: Invalid unit: X
Temperature(-300, 'C'): ValueError: Temperature below absolute zero
Temperature(-500, 'F'): ValueError: Temperature below absolute zero
Temperature(-1, 'K'): ValueError: Temperature below absolute zero
t.value = 30: AttributeError: property 'value' of 'Temperature' object has no setter


### Comparisons across units

In [4]:
print("0 °C == 32 °F       :", Temperature(0, "C") == Temperature(32, "F"))
print("0 °C == 273.15 K    :", Temperature(0, "C") == Temperature(273.15, "K"))
print("-40 °C == -40 °F    :", Temperature(-40, "C") == Temperature(-40, "F"))
print("37 °C < 100 °F      :", Temperature(37, "C") < Temperature(100, "F"))
print("Temperature == 0    :", Temperature(0, "C") == 0)

mixed = [Temperature(100, "C"), Temperature(0, "K"), Temperature(32, "F"), Temperature(310.15, "K")]
print("Sorted:", [str(x) for x in sorted(mixed)])

0 °C == 32 °F       : True
0 °C == 273.15 K    : True
-40 °C == -40 °F    : True
37 °C < 100 °F      : True
Temperature == 0    : False
Sorted: ['0.00 K', '32.00 °F', '310.15 K', '100.00 °C']


### Test results

In [5]:
!{sys.executable} -m pytest -v --color=no test_temperature.py

============================= test session starts ==============================
platform linux -- Python 3.12.3, pytest-9.2.0, pluggy-1.7.0 -- /home/kristian/Coding/INF201/Oblig/inf201_deliverable_4/.venv/bin/python
cachedir: .pytest_cache
rootdir: /home/kristian/Coding/INF201/Oblig/inf201_deliverable_4
plugins: platformdirs-4.14.8
collected 30 items                                                             

test_temperature.py::test_default_unit PASSED                            [  3%]
test_temperature.py::test_case_insensitive_unit[c] PASSED                [  6%]
test_temperature.py::test_case_insensitive_unit[f] PASSED                [ 10%]
test_temperature.py::test_case_insensitive_unit[k] PASSED                [ 13%]
test_temperature.py::test_case_insensitive_unit[C] PASSED                [ 16%]
test_temperature.py::test_case_insensitive_unit[F] PASSED                [ 20%]
test_temperature.py::test_case_insensitive_unit[K] PASSED                [ 23%]
test_temperature.py::tes

## Task 2: `Fraction`

**Attributes:** `numerator`, `denominator`<br>
**Special methods:**
* Initialiser: `__init__`
* Comparisons: `__eq__`, `__ne__`, `__lt__`, `__le__`, `__gt__`, `__ge__`
* Other: `__add__`, `__sub__`, `__mul__`, `__truediv__`, `__float__`, `__repr__`, `__str__`

Design choices:
* Both parts must be `int`, otherwise `TypeError`. A zero denominator raises `ZeroDivisionError`.
* A negative denominator is moved to the numerator, so the denominator is always strictly positive.
* Every fraction is reduced with `math.gcd`, so `Fraction(2, 4)` and `Fraction(1, 2)` are stored the same way.
* Comparisons cross-multiply (`a/b < c/d` ⇔ `a·d < c·b`, valid since denominators are positive), so they are exact with no floating-point error.
* Integers are accepted as the other operand (`Fraction(1, 2) + 1`). Any other type returns `NotImplemented`.

### Creating fractions

In [6]:
for args in [(3, 4), (2, 4), (2, -4), (6, 3), (5,)]:
    f = Fraction(*args)
    print(f"Fraction{args!s:8} -> str: {f!s:5}  repr: {f!r}")

Fraction(3, 4)   -> str: 3/4    repr: Fraction(3, 4)
Fraction(2, 4)   -> str: 1/2    repr: Fraction(1, 2)
Fraction(2, -4)  -> str: -1/2   repr: Fraction(-1, 2)
Fraction(6, 3)   -> str: 2/1    repr: Fraction(2, 1)
Fraction(5,)     -> str: 5/1    repr: Fraction(5, 1)


### Invalid input

In [7]:
for args in [(1, 0), (1.5, 2), ("1", 2)]:
    try:
        Fraction(*args)
    except (TypeError, ZeroDivisionError) as error:
        print(f"Fraction{args}: {type(error).__name__}: {error}")

Fraction(1, 0): ZeroDivisionError: Denominator cannot be zero
Fraction(1.5, 2): TypeError: Numerator and denominator must be integers
Fraction('1', 2): TypeError: Numerator and denominator must be integers


### Arithmetic and conversion

In [8]:
a, b = Fraction(1, 2), Fraction(1, 3)
print(f"{a} + {b} = {a + b}")
print(f"{a} - {b} = {a - b}")
print(f"{a} * {b} = {a * b}")
print(f"{a} / {b} = {a / b}")
print(f"{a} + 1 = {a + 1}")
print(f"float({b}) = {float(b)}")

1/2 + 1/3 = 5/6
1/2 - 1/3 = 1/6
1/2 * 1/3 = 1/6
1/2 / 1/3 = 3/2
1/2 + 1 = 3/2
float(1/3) = 0.3333333333333333


### Comparisons

In [9]:
a, b = Fraction(1, 2), Fraction(2, 3)
print(f"{a} == 2/4 : {a == Fraction(2, 4)}")
print(f"{a} != {b} : {a != b}")
print(f"{a} <  {b} : {a < b}")
print(f"{a} <= {b} : {a <= b}")
print(f"{a} >  {b} : {a > b}")
print(f"{a} >= {b} : {a >= b}")
print(f"4/2 == 2   : {Fraction(4, 2) == 2}")

print("Sorted:", [str(x) for x in sorted([Fraction(3, 4), Fraction(-1, 2), Fraction(2, 3), Fraction(1, 8)])])

1/2 == 2/4 : True
1/2 != 2/3 : True
1/2 <  2/3 : True
1/2 <= 2/3 : True
1/2 >  2/3 : False
1/2 >= 2/3 : False
4/2 == 2   : True
Sorted: ['-1/2', '1/8', '2/3', '3/4']


### Test results

In [10]:
!{sys.executable} -m pytest -v --color=no test_fraction.py

============================= test session starts ==============================
platform linux -- Python 3.12.3, pytest-9.2.0, pluggy-1.7.0 -- /home/kristian/Coding/INF201/Oblig/inf201_deliverable_4/.venv/bin/python
cachedir: .pytest_cache
rootdir: /home/kristian/Coding/INF201/Oblig/inf201_deliverable_4
plugins: platformdirs-4.14.8
collected 7 items                                                              

test_fraction.py::test_init_reduces_and_normalises_sign PASSED           [ 14%]
test_fraction.py::test_init_default_denominator PASSED                   [ 28%]
test_fraction.py::test_init_invalid PASSED                               [ 42%]
test_fraction.py::test_comparisons PASSED                                [ 57%]
test_fraction.py::test_arithmetic PASSED                                 [ 71%]
test_fraction.py::test_float PASSED                                      [ 85%]
test_fraction.py::test_str_and_repr PASSED                               [100%]

=======================

## GitHub, GenAI use and video

* **Repository:** <https://github.com/SheathAndWhitley/inf201_deliverable_4>
* **Commit log:** `git_log.txt`
* **Virtual environment:** `requirements.txt`
* **GenAI use:** see `GenAI_Use.md`
* **Video:** *TODO – add link or file name*